## Hyperbolic Distances

This notebook:
- Loads ENAO hyperbolic embeddings
- Computes Lorentz distances for collaboration pairs across all years (2017-2019)
- Normalizes distances (applies min-max scaling to bring raw distances into [0, 1])
- Saves combined dataset with both raw and normalized distances
- Saves embeddings for genres present in both the ENOA vocabulary and the collaboration network (2017–2019)

In [1]:
import os
os.chdir('..') 
print("Now working from:", os.getcwd())

Now working from: /Users/mikolajandrzejewski/Documents/GitHub/Bachelor


In [2]:
import pickle
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sys.path.append("embeddings")
from lorentz import Lorentz

/Users/mikolajandrzejewski/anaconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (6.0.0.post1)/charset_normalizer (2.1.1) doesn't match a supported version!
  warnings.warn(


enao_graph


## 1. Load ENAO embeddings and vocabulary

In [3]:
vocab = pickle.load(open("enao_vocab.pkl", "rb"))
genre_to_idx = {genre: idx for idx, genre in enumerate(vocab)}
checkpoint = torch.load("ckpt/final_enao_graph.ckpt", map_location="cpu")
embeddings = checkpoint["table.weight"].numpy()

print(f"ENAO vocabulary size: {len(vocab)} genres")
print(f"Embedding shape: {embeddings.shape}")

ENAO vocabulary size: 6280 genres
Embedding shape: (6281, 3)


## 2. Define Lorentz distance function

In [4]:
def lorentz_scalar_product(u, v):
    """Compute Lorentz scalar product."""
    return -u[0]*v[0] + np.dot(u[1:], v[1:])

def lorentz_distance(u, v):
    """Compute hyperbolic distance in Lorentz model."""
    inner = lorentz_scalar_product(u, v)
    inner = min(inner, -1.0)  
    return float(np.arccosh(-inner))

## 3. Process all years (2017-2019)

In [5]:
years = [2017, 2018, 2019]
all_dfs = []
all_matched_genres = set()
all_unmatched_genres = set()

for year in years:
    print("\n" + "="*60)
    print(f"Year: {year}")

    # Load collaboration dataset for this year
    df = pd.read_csv(f"datasets/Original/global/global-genre_network-{year}.csv", sep="\t")
    
    # Remove self-loops
    df = df[df["source"] != df["target"]].copy()
    collab_genres = set(df["source"].unique()) | set(df["target"].unique())
    print(f"Unique genres in collaboration dataset: {len(collab_genres)}")
    
    # Match genres with ENAO embeddings
    matched = {}
    unmatched = []
    for genre in collab_genres:
        if genre in genre_to_idx:
            idx = genre_to_idx[genre] + 1  
            matched[genre] = embeddings[idx]
            all_matched_genres.add(genre)
        else:
            unmatched.append(genre)
            all_unmatched_genres.add(genre)
    
    print(f"Matched: {len(matched)} genres")
    if unmatched:
        print(f"Unmatched genres ({len(unmatched)} genres):")
        for g in sorted(unmatched):
            print(f"  - {g}")
    
    # Compute distances for each collaboration pair
    distances = []
    skipped_rows = []
    for _, row in df.iterrows():
        source = row["source"]
        target = row["target"]
        
        if source not in matched:
            skipped_rows.append((source, target, f"'{source}' not in ENAO"))
            distances.append(None)
            continue
        if target not in matched:
            skipped_rows.append((source, target, f"'{target}' not in ENAO"))
            distances.append(None)
            continue
        
        d = lorentz_distance(matched[source], matched[target])
        distances.append(d)
    
    df["distance"] = distances
    df["year"] = year
    
    # Remove rows without distances
    df_clean = df.dropna(subset=["distance"]).copy()
    
    all_dfs.append(df_clean)

df_combined = pd.concat(all_dfs, ignore_index=True)

print("\n" + "="*60)
print("COMBINED DATASET SUMMARY")
print(f"Total unique genres with embeddings: {len(all_matched_genres)}")
print(f"Total unique genres without embeddings: {len(all_unmatched_genres)}")
print(f"Unmatched genres across all years:")
for g in sorted(all_unmatched_genres):
    print(f"  - {g}")


Year: 2017
Unique genres in collaboration dataset: 258
Matched: 248 genres
Unmatched genres (10 genres):
  - afro dancehall
  - baile pop
  - deep pop r&b
  - francoton
  - indie cafe pop
  - kwaito house
  - latin
  - lgbtq+ hip hop
  - ninja
  - vapor trap

Year: 2018
Unique genres in collaboration dataset: 295
Matched: 284 genres
Unmatched genres (11 genres):
  - afro dancehall
  - baile pop
  - deep pop r&b
  - francoton
  - indie cafe pop
  - kwaito house
  - latin
  - lgbtq+ hip hop
  - nc hip hop
  - ninja
  - vapor trap

Year: 2019
Unique genres in collaboration dataset: 310
Matched: 297 genres
Unmatched genres (13 genres):
  - afro dancehall
  - baile pop
  - deep pop r&b
  - dutch urban
  - francoton
  - k-hop
  - latin
  - lgbtq+ hip hop
  - nc hip hop
  - ninja
  - regional mexican pop
  - trap espanol
  - vapor trap

COMBINED DATASET SUMMARY
Total unique genres with embeddings: 391
Total unique genres without embeddings: 15
Unmatched genres across all years:
  - afro danc

## 4. Normalize distances

In [6]:
distance_stats = df_combined['distance'].describe()
print(f"\nRange (raw): min={distance_stats['min']:.4f}, max={distance_stats['max']:.4f}")

# Min-Max normalization (0-1 range)
min_max_scaler = MinMaxScaler()
df_combined['distance_norm'] = min_max_scaler.fit_transform(df_combined[['distance']])

print(f"Range (normalized): min={df_combined['distance_norm'].min():.4f}, max={df_combined['distance_norm'].max():.4f}")


Range (raw): min=0.0221, max=8.9440
Range (normalized): min=0.0000, max=1.0000


## 5. Save combined dataset

In [7]:
column_order = ['year', 'source', 'target', 'weight', 'avg_streams', 
                'distance', 'distance_norm']
df_combined = df_combined[column_order]

output_file = "exploratory_analysis/collaboration_with_distances.csv"
df_combined.to_csv(output_file, index=False)

print(f"Combined dataset saved to: {output_file}")

Combined dataset saved to: exploratory_analysis/collaboration_with_distances.csv


## 6. Save matched genre embeddings

In [8]:
final_matched = {}
for genre in all_matched_genres:
    idx = genre_to_idx[genre] + 1
    final_matched[genre] = embeddings[idx]

pickle.dump(final_matched, open("exploratory_analysis/collab_genre_embeddings.pkl", "wb"))
print(f"Saved {len(final_matched)} genre embeddings to 'collab_genre_embeddings.pkl'")

Saved 391 genre embeddings to 'collab_genre_embeddings.pkl'
